## Data-Driven Parking Management: Assessing Utilization and Revenue in Pittsbrugh
95885 - Project 1 Final Report
Name: Andrew Huang (tzuchua2), Bohua Liu (bohual)


## Executive Summary
This project analyzes parking usage and revenue efficiency across Pittsburgh using city parking transaction and infrastructure datasets.

#### Key findings:
*   We are able to build a parking recommendation program using government (City of Pittsburgh) datasets that suggests the best nearby parking locations based on your destination and your preferences for price, walking distance, and availability (busyness).
*  Parking prices are generally positively related to busyness—higher-priced zones tend to be busier—but the relationship is imperfect, with clear “mismatch” zones where similar prices have very different demand.
*   Among parking areas with demand, average daily revenue per parking space varies from about \$0.26 per space per day at Observatory Hill Lot to about $1.76 at Ansley Beatty Lot. This indicates more than a 6× difference in parking revenue efficiency across locations.

#### Recommendations:
1. Suppose we are going to class at Hamburg Hall (HBH). If you are demand-averse like me—meaning your top priority is finding a spot with a higher chance of availability—then the best “go-first” parking locations are: Morning: Schenley Dr → Schenley Dr Ext → S Craig St; Afternoon: S Craig St → Filmore St → Winthrop St.
2. Use these specific zones as targets for pricing/rule adjustments—consider raising rates or tightening time limits for the over-busy zones (Carson lots; Ivy Bellefonte; Forbes Shady) to reduce cruising and improve turnover, and lowering rates, revising restrictions, or improving wayfinding for the under-busy zones (Brookline, Beechview, Ervatory Hill; Bakery Sq; Douglas Phillips) to increase utilization—then recheck transactions-per-space to confirm the mismatch shrinks.
3. The parking authority should prioritize expanding or reallocating parking supply in high-efficiency areas such as Ansley Beatty Lot and Mt. Washington, while reevaluating or reducing excess parking capacity in low-efficiency areas such as Observatory Hill Lot.

## Problem Context
In dense urban environments, parking is often a major challenge and in a city that has so many hills, rivers, and bridges, it's a even a more challenging task for the Pittsburgh government to design and build parking lots that maximize convenience for local residents. There are around 854,000 vehicles commuting in the city daily and due to such large amount of commuters, parking can be a real headache to anyone of us. As a driver myself, I oftentimes arrive at a parking lot and figured it's full, which I need to spend a lot of time finding an empty lot nearby and often eventually failed to appear. As a result, we expect our solution to make Pittsburgh drivers' lives more convenient by saving their time to search for parking lots and even save their money if they forget to pay. To make sure our recommendations are valuable and plauisble, our target audience would be the Department of Urban Planning as well as the Pittsburgh Parking Authority, who have actual rights to modify policies related to parking.

After some preliminary research, we have found out that the Pittsburgh parking system has a few inefficiencies that can be improved. Firstly, there's limited physical availability. Pittsburgh is unlike from NYC, where there are too many skyscrapers and makes every space expensive. In fact, it's the geographical characteristics of rivers, hills, mountains, and bridges that limit he physical availability. This causes most of the parking lots to be small and thus making drivers spending more time searching for a suitable parking lot.

The goal of this project is to analyze Pittsburgh parking data to better understand parking demand, utilization, and revenue efficiency. Using five final datasets provided by the city after cleaning and normalizing, we investigated how parking spaces are used across different areas and identify opportunities for improving parking management decisions.

During the project we aim to answer these three questions below and we have attached our recommendations below them.

## Research Questions
To help us formulate specific recommendations to the Pittsburgh Parking Authority, our project focus on several reserach questions as a guide for our analysis directions:


1.   Where should I park near a target place?
2.   Are parking prices related to how busy an area is?
3.   Which parking areas generate the least revenue per parking space?





## Data Description

### Data Sources
The datasets used in this project were obtained from the City of Pittsburgh’s public parking meter data portal. They cover parking infrastructure, pricing, space counts, and payment transactions, and represent the most complete publicly available sources for supporting our analysis.

The raw datasets downloaded from the government portal include:
*   Parking Meter Information.csv - metadata about individual parking meters.
*   Payment Transactions across meters.csv – paid parking activity time-series in 10-minute bins, by zone and payment channel (meter vs mobile), 2014–2019.
*   Space Counts and Rates.csv – information about the number of parking spaces and pricing rates.

These datasets provide the foundation for analyzing parking supply, usage, and revenue patterns across Pittsburgh.

However, because they were published as three independent tables **without a shared, ready-to-use join key**, we must apply data manipulation to create appropriate connections so that meters, zones, transactions, and space/rate attributes can be merged consistently for analysis.

### Data Cleaning
First, we performed data cleaning on single datasets: we **removed unnecessary columns** not relevant to the research questions and **reshaped the data into a tidy format** (variables in columns, observations in rows). (See *parking_meter_data.ipynb*, *payment_data.ipynb*, *space_counts_rates.ipynb* for further details)

Second, because the three datasets are stored independently and do not share a clean built-in key, we **normalized column names and formats across datasets** to enable reliable merges. In particular, parking zone identifiers were normalized so that parking transaction data could be linked with parking space and location information. (See *datasets_join.ipynb* for futher details)

Finally, we split the cleaned files into **observational unit-specific tables** (e.g., parking meter data.csv → parking_zones, parking_rates, parking_meters) and stored them in a final dataset directory to make analysis and joins clearer. (See *parking_meter_data.ipynb, section '3. Tidy Data'*, *payment_data.ipynb, section '3. Tidy Data'* for further details)


Since CSV cannot preserve Polars datetime types, we **convert the transaction timestamps** to proper datetime formats each time we load the data to enable time-based aggregation. (See *datasets_join.ipynb, section '3. Saving final datasets'* for futher details)

### Final Datasets Used
After cleaning and normalization, the following datasets were used for the analysis:
* parking_meters.csv - Metadata for individual parking meters, including meter identifiers, associated zone info, street/location labels, and coordinates (x/y).
* parking_rates.csv - Meter-level pricing rules, including the hourly rate and the time window(s) when each rate applies (some meters have multiple rate windows).
* parking_zones.csv - Zone-level reference data for Pittsburgh parking areas, including zone identifiers and zone names/labels. This dataset is used to group individual meters into comparable parking zones and serves as a key link for joining meter, pricing, space, and transaction data at the zone level.
*   parking_spaces_zone_group_norm.csv - Provides the number of parking spaces available in each parking zone and the type of parking infrastructure (on-street or off-street).
*   payment_transactions.csv - Contains records of individual parking payment transactions, including payment amounts and timestamps. This dataset is used to measure parking demand and revenue.



### Constraints & Assumptions
However, a key constraint is that the datasets are not time-aligned: the payment transactions span 2014-2019, the space data covers only 2017-2018, and the parking zones/rates tables appear to be a latest snapshot updated March 11, 2026. As a result, any analysis that joins these sources must assume that zone definitions and related attributes (e.g., rates and space inventory), as well as transaction patterns, remained stable across these periods, or at least changed in ways that do not materially affect zone-level patterns and comparisons.

In addition, because we joined three independent datasets using a custom normalized key, some zones did not match across sources and were excluded from downstream analysis. Therefore, we assume that the normalized join key correctly preserves the intended meaning of zone identifiers and that the retained matched zones are representative for zone-level comparisons. This assumption is reasonable because the normalized key is derived from (and preserves) the original zone location, so the matched zones still correspond to the same underlying spatial units. (See *datasets_join.ipynb, section '1. Parking Zones & Payment Transactions' and section '2. Payments Transaction & Space Counts'* for further details)

Throughout the analysis, we use **transactions per space** as a proxy for busyness rather than a direct measure of real-time occupancy, since occupancy data are not available. This means the measure captures how frequently spaces are used over time, but it may not fully reflect actual parking congestion at any given moment.

## Question 1: Where should I park near a target place?

We want to use the City of Pittsburgh parking datasets to rank and recommend nearby parking zones, helping drivers choose parking more efficiently and reduce time spent searching for a spot.

This is really cool because you can plug in any destination (any “X,Y” point) in Pittsburgh, and the program will generate a personalized parking recommendation based on your preferences — so the “best” option adapts to what you care about most.

### 1.1. Method
We solve this problem along three dimensions: **price**, **availability**, and **distance**, and The final recommendation is a ranked list based on a weighted combination of these three dimensions.

To compute these dimensions for each nearby parking meter, we first derive a few practical variables—meter **location**, **capacity**, **active hours**, and **observed demand**—from the Pittsburgh parking CSVs:

- **Location (x/y of each meter):** from **parking_meters.csv**  
- **Capacity (spaces per zone):** from **parking_spaces_zone_group_norm.csv**  
- **Price (hourly rate):** from **parking_rate.csv**, joined to meters to assign a rate to each candidate meter   
- **Demand / busyness (usage level):** from **payment_transactions.csv**, counting transactions during active hours  
  - then compute an availability proxy such as **transactions_per_space = total_transactions / total_spaces**
- **Distance to destination:** computed from meter coordinates in **parking_meters.csv** using **haversine**

After computing these quantities for all candidate meters near the destination, we normalize them and combine them with user-defined weights into a single score, producing the final ranked parking recommendations.

### 1.2. Constraint / Assumption

Transaction counts and space capacity are only available at the **zone level**, not per individual meter.  
To estimate meter-level values, we assume demand and capacity are uniformly distributed within each zone.

- **Meter-level spaces:**
$$
\mathrm{spaces\_per\_meter}=\frac{\mathrm{total\_spaces\_in\_zone}}{\mathrm{\#\ meters\ in\ zone}}
$$

- **Meter-level transactions** (within the chosen time window and active hours):
$$
\mathrm{transactions\_per\_meter}=\frac{\mathrm{total\_transactions\_in\_zone}}{\mathrm{\#\ meters\ in\ zone}}
$$

We then compute the busyness proxy at the meter level using these evenly divided estimates.  

Another assumption is about distance. We measure distance using the **Haversine formula**, which gives the **straight-line distance** between two coordinates. So we assume that this is a reasonable approximation of the actual walking distance. It may not be exactly the same as the route someone would walk, but it gives us a simple and consistent way to compare nearby parking options.

### 1.3. Code

The code here is designed to provide a clearer overview of the notebook. For more detailed, low-level information, please see ./detailed_analysis/where_park.ipynb.

**Imports & Helper Function**

We don't have distance information in datasets but we do have x, y coordinates. Thus, we could use Haversine formula to calculate distance.

Reference: https://en.wikipedia.org/wiki/Haversine_formula

In [ ]:
import polars as pl
import altair as alt
import numpy as np

# ----------------
# DATA IMPORTS
# ----------------
parking_rates = pl.read_csv("final_datasets/parking_rates.csv")
parking_meters = pl.read_csv("final_datasets/parking_meters.csv")
# parking spaces (zone level)
parking_spaces_zone_group_norm = pl.read_csv("final_datasets/parking_spaces_zone_group_norm.csv")
# payment_transactions (zone level)
payment_transactions = pl.read_csv("final_datasets/payment_transactions.csv")
# used to join zone-level results to other parking tables keyed by zone_id
# i.e. parking_xxx.csv
zone_group_norm_to_zone_id = pl.read_csv("final_datasets/zone_group_norm_to_zone_id.csv")

# define joinable zone list
joinable_zones_lst = (
    parking_rates.join(zone_group_norm_to_zone_id, on="zone_id")
    # we have one to many zone_group_norm - zone_id mapping. See datasets_join.ipynb for details
    .unique(subset="zone_group_norm",keep="first")
    .join(parking_spaces_zone_group_norm, on="zone_group_norm")
    .join(payment_transactions, on="zone_group_norm")
    # some zones cannot be normalized to allow joining between different datasets
    # so they are not joinable. See datasets_join.ipynb for details
    .select("zone_group_norm")
    .unique().to_series().to_list()
)

# ---------------------------------------------------
# HELPER FUNC: Haversine formula distance calculation
# ---------------------------------------------------
def haversine_m(lon1, lat1, lon2, lat2):
    """
    Compute the great-circle (Haversine) distance between two points on Earth.

    Parameters
    lon1, lat1 : float or array-like
        Longitude and latitude of the first point in decimal degrees.
    lon2, lat2 : float or array-like
        Longitude and latitude of the second point in decimal degrees.

    Returns
    float or np.ndarray
        Great-circle distance between the two points in **meters**.
        The return type matches the input type: scalar inputs return a float,
        array-like inputs return a NumPy array.
    """
    R = 6371000.0
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)

    a = np.sin(dphi/2.0)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dl/2.0)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))


**Variable Preparation**

In [ ]:
# ------------------------
# Distance Calculation
# ------------------------
meters = (
    parking_meters
    .join(zone_group_norm_to_zone_id, on="zone_id")
    .filter(
        pl.col("zone_group_norm").is_in(joinable_zones_lst)
    )
)
# target destination's (x,y)
CMU_x, CMU_y = -79.9457643, 40.4443039

# radius choices (meters)
RADIUS_M = 800   # ~0.5 mile

meters_near = (
    meters
    .with_columns(
        pl.struct(["x", "y"]).map_elements(
            lambda s: float(haversine_m(s["x"], s["y"], CMU_x, CMU_y))
            # lambda s: float(haversine_m(s["x"], s["y"], PANDA_x, PANDA_y))
        ).alias("dist_m")
    )
    .filter(pl.col("dist_m") <= RADIUS_M)
    .sort("dist_m")
)


# ------------------------------------------
# Joining Meters, Rate, Transaction, Spaces
# ------------------------------------------
# build zone totals
zone_totals = (
    payment_transactions
    .group_by("zone_group_norm")
    .agg([
        pl.col("n_transaction").sum().alias("zone_transactions"),
    ])
)

# count meters per zone
meters_per_zone = (
    meters_near
    .select(["zone_group_norm", "meter_id"])
    .unique()
    .group_by("zone_group_norm")
    .len()
    .rename({"len": "n_meters_zone"})
)

# join spaces + zone transactions to meters, then allocate
meters_rates_spaces_tx_near = (
    meters_near.select(['meter_id', 'location', 'x', 'y', 'dist_m','zone_group_norm'])
    .join(parking_rates, on="meter_id")
    .join(parking_spaces_zone_group_norm, on="zone_group_norm", how="left")
    .join(zone_totals, on="zone_group_norm", how="left")
    .join(meters_per_zone, on="zone_group_norm", how="left")
    # allocate zone-level values down to meter level (uniform split assumption)
    .with_columns([
        (pl.col("spaces") / pl.col("n_meters_zone")).alias("spaces_per_meter"),
        (pl.col("zone_transactions") / pl.col("n_meters_zone")).alias("tx_per_meter"),
    ])
    .select([
        'meter_id', 'location', 'x', 'y', 'dist_m', 'zone_group_norm',
        'rate_window', 'rate_time', 'rate', 'type',
        'zone_transactions', 'spaces_per_meter', 'tx_per_meter'
    ])
)

# rate time check
meters_rates_spaces_tx_near.select("rate_time").unique()

rate_time
str
"""~ 8AM - 2PM"""
"""~ 2PM - 6PM"""
"""all time"""


In [ ]:
# Divede dataset to morning (8AM - 2PM) and afternoon (2PM - 6PM)
morning_analysis = (
    meters_rates_spaces_tx_near
    .filter(
        pl.col("rate_time") != "~ 2PM - 6PM"
    )
)
afternoon_analysis = (
    meters_rates_spaces_tx_near
    .filter(
        pl.col("rate_time") != "~ 8AM - 2PM"
    )
)

**Parking locaiton ranking plot**

In [ ]:
# ------------
# HELPER FUNC
# ------------
def location_summary(df, period):
    """
    Aggregate meter-level observations into a location-level summary table.

    Parameters
    df : pl.DataFrame
        Meter-level data containing (at minimum) columns:
        ['location', 'x', 'y', 'rate', 'tx_per_meter', 'spaces_per_meter', 'dist_m', 'meter_id'].
        Each row typically represents a meter observation (or a meter-rate record) near the destination.
    period : str
        Label for the time slice being summarized (e.g., "Morning", "Afternoon").

    Returns
    pl.DataFrame
        One row per `location` with representative price, demand/capacity totals,
        distance summaries, and a derived busyness proxy `transactions_per_space`.
    """
    return (
        df.group_by("location")
        .agg([
            pl.col("x").min().alias("min_x"),
            pl.col("y").min().alias("min_y"),
            pl.col("rate").median().alias("rate"),
            pl.col("tx_per_meter").sum().alias("tot_transactions"),
            pl.col("spaces_per_meter").sum().alias("tot_spaces"),
            pl.col("dist_m").min().alias("min_dist_m"),
            pl.col("dist_m").median().alias("median_dist_m"),
            pl.col("meter_id").n_unique().alias("n_meters"),
        ])
        .with_columns(
            (pl.col("tot_transactions") / pl.col("tot_spaces")).alias("transactions_per_space"),
            pl.lit(period).alias("period")
        )
    )

def minmax_norm(col: str) -> pl.Expr:
    """
    Create a Polars expression that min-max normalizes a column to the range [0, 1].

    Normalization formula:
        (x - min(x)) / (max(x) - min(x))

    Edge case:
    If max(x) == min(x), the denominator becomes 0 (no variation). In that case,
    this function returns 0.5 for all rows to represent a neutral midpoint.

    Parameters
    col : str
        Column name to normalize.

    Returns
    pl.Expr
        Polars expression producing a normalized column in [0, 1].
    """
    denom = pl.col(col).max() - pl.col(col).min()
    return (
        pl.when(denom == 0)
        .then(pl.lit(0.5))
        .otherwise((pl.col(col) - pl.col(col).min()) / denom)
    )

def add_score(df, w_price=0.3, w_busy=0.2, w_dist=0.5):
    """
    Add normalized features and a preference-weighted score, then return a ranked table.

    The score is a weighted sum of three normalized components (lower is better):
        score = w_price * price_norm
              + w_busy  * busy_norm
              + w_dist  * dist_norm

    Where:
    - price_norm is min-max normalized `rate`
    - busy_norm  is min-max normalized `transactions_per_space`
    - dist_norm  is min-max normalized `min_dist_m`

    Parameters
    df : pl.DataFrame
        Location-level summary table containing columns:
        ['rate', 'transactions_per_space', 'min_dist_m'] at minimum.
    w_price : float, default 0.3
        Weight for normalized price (rate).
    w_busy : float, default 0.2
        Weight for normalized busyness (transactions per space).
    w_dist : float, default 0.5
        Weight for normalized distance (minimum distance in meters).

    Returns
    pl.DataFrame
        Input table with added columns:
        ['price_norm', 'busy_norm', 'dist_norm', 'score'],
        sorted ascending by `score` (best recommendations first).
    """
    return (
        df.with_columns([
            minmax_norm("rate").alias("price_norm"),
            minmax_norm("transactions_per_space").alias("busy_norm"),
            minmax_norm("min_dist_m").alias("dist_norm"),
        ])
        .with_columns(
            (w_price*pl.col("price_norm") + w_busy*pl.col("busy_norm") + w_dist*pl.col("dist_norm")).alias("score")
        )
        .sort("score")
    )


def recommendation_bar(df, title, width=330, height=350):
    """
    Create a bar chart of ranked parking recommendations.

    Parameters
    df : pandas.DataFrame
        Ranked recommendation table containing at least:
        ['location', 'score', 'rate', 'min_x', 'min_y',
         'transactions_per_space', 'tot_transactions', 'tot_spaces',
         'min_dist_m', 'median_dist_m', 'n_meters'].
    title : str
        Chart title.
    width : int, default 330
        Chart width in pixels.
    height : int, default 350
        Chart height in pixels.

    Returns
    alt.Chart
        Altair bar chart sorted by score (lower = better).
    """
    return (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X("score:Q", title="Score (lower = better)"),
            y=alt.Y(
                "location:N",
                sort=alt.SortField(field="score", order="ascending"),
                title="Location"
            ),
            color=alt.Color("score:Q", title="Score"),
            tooltip=[
                "location",
                alt.Tooltip("rate:Q", format=".2f", title="Rate"),
                alt.Tooltip("score:Q", format=".2f", title="Score"),
                alt.Tooltip("min_x:Q", format=".6f", title="Closest x"),
                alt.Tooltip("min_y:Q", format=".6f", title="Closest y"),
                alt.Tooltip("transactions_per_space:Q", format=".3f", title="Tx per space (busyness)"),
                alt.Tooltip("tot_transactions:Q", format=",.0f", title="Total transactions"),
                alt.Tooltip("tot_spaces:Q", format=",.0f", title="Total spaces"),
                alt.Tooltip("min_dist_m:Q", format=".2f", title="Min dist (m)"),
                alt.Tooltip("median_dist_m:Q", format=".2f", title="Median dist (m)"),
                alt.Tooltip("n_meters:Q", format=",.0f", title="# meters"),
            ]
        )
        .properties(width=width, height=height, title=title)
    )



# build location summaries for different time slices
loc_m = location_summary(morning_analysis, "Morning")
loc_a = location_summary(afternoon_analysis, "Afternoon")

# user preference example: prioritize availability heavily
w_price, w_busy, w_dist = 0.2, 0.7, 0.1

# ranked recommendations for each period
rank_m = add_score(loc_m, w_price, w_busy, w_dist)
rank_a = add_score(loc_a, w_price, w_busy, w_dist)

# take top-N and convert to pandas for Altair plotting
m_top = rank_m.head(10).to_pandas()
a_top = rank_a.head(10).to_pandas()

# usage
c1 = recommendation_bar(m_top, "8AM - 2PM: Recommended locations")
c2 = recommendation_bar(a_top, "2PM - 6PM: Recommended locations")

(c1 | c2).resolve_scale(x="shared", color="shared")

alt.HConcatChart(...)

### 1.4. Interpretation of the ranking (Morning vs Afternoon)

These plots show the **top recommended parking locations** for two time periods. Each bar represents a location, and the bar length/color is its **final score** (lower = better).

#### Weights used (preference setting)
The score is computed from normalized features using:
- **Price weight (w_price) = 0.2**
- **Busyness / availability weight (w_busy) = 0.7**
- **Distance weight (w_dist) = 0.1**

So the ranking is **mostly driven by availability (low transactions per space)**, with price having a smaller effect and distance only a minor effect.

#### Morning (8AM - 2PM) recommendations
In the **morning**, locations like **SCHENLEY DR** and **SCHENLEY DR EXT** appear at the top (lowest scores).  
This suggests that these areas are estimated to be **less busy per space** during the morning window, even if some other locations may be slightly cheaper or closer.

#### Afternoon (2PM - 6PM) recommendations
In the **afternoon**, the top-ranked locations shift: **S CRAIG ST**, **FILMORE ST**, and **WINTHROP ST** are now best (lowest scores).  
This indicates the demand pattern changes over time—these locations appear **less busy relative to capacity** in the afternoon compared to others.

How ever, the drop in ranking for **SCHENLEY DR / SCHENLEY DR EXT** in the afternoon is primarily explained by a **higher afternoon rate (price increases)**, rather than a clear increase in the busyness proxy.


#### Takeaway
Because we set **w_busy = 0.7**, the model is effectively answering:
> “Where is most likely to have an available spot (least busy), even if it isn’t the absolute cheapest or closest?”

The main insight from the plots is that **the best parking locations depend strongly on time of day**, and a demand-aware approach can surface different “best” streets for morning vs afternoon arrivals.

## Question 2: Are parking prices related to how busy an area is?

### 2.1. Why this question is valuable
Understanding whether **parking prices reflect how busy an area is** helps evaluate if pricing is working as a demand-management tool.

If parking prices **match** how busy an area is, then:

- **Higher demand → higher price → fewer people choose to park there → more turnover → better chance of finding a space.**
- **Lower demand → lower price → attracts drivers to underused areas → balances parking usage across the city.**
- **Prices aligned with demand → revenue reflects usage → more funding for maintenance and transport.**

If prices **don’t** match busyness, then:

- **Busy area + low price → too many drivers → full spaces → people circle longer → more congestion.**
- **Quiet area + high price → few drivers → empty spaces → wasted supply and lower economic activity nearby.**
- **Mismatch → pricing feels unfair and policy becomes less effective.**

### 2.2. Method
We treat this as a **relationship analysis** between two zone-level variables: **price** and **busyness**.

To compute these, we first construct our two zone-level variables from the parking tables:

- **Define the analysis unit (zones)**  
   Use **parking_zones.csv** to identify and label zones, and use the zone identifiers needed to join across datasets.

- **Compute zone-level price**  
   Use **parking_rates.csv** to get meter hourly rates, then aggregate within each zone (**mean rate**) to represent the typical price level.

- **Compute zone-level busyness (transactions per space)**  
   - Sum total transactions per zone from **payment_transactions.csv**.  
   - Get total parking capacity per zone from **parking_spaces_zone_group_norm.csv**.  
   - Compute the busyness proxy (zone-level):  
 $$
  \mathrm{transactions\_per\_space}
  =
  \frac{\mathrm{total\_transactions}}{\mathrm{total\_spaces}}
  $$
Once we have these zone-level variables, we analyze how they relate:

- **Compare price vs. busyness (visual check)**  
  Create bar charts that rank zones by **mean rate** and by **transactions\_per\_space**, then compare the patterns side-by-side.  
  If pricing reflects demand, zones that appear near the top of the busyness ranking should also tend to appear near the top of the price ranking.

- **Diagnose mismatches (actionable cases)**  
  Identify zones that fall into “unexpected” categories such as **busy-but-cheap** (high busyness, low price) or **quiet-but-expensive** (low busyness, high price).  
  These mismatched zones are treated as candidates for further policy review—potentially involving pricing updates, enforcement-hour changes, or adjustments to supply constraints.

### 2.3. Assumptions and limitations

Both **price** and **busyness** are measured at the **zone level**. This simplifies comparison, but it can hide important variation within a zone, since some meters or blocks may be much busier than others.

For **price**, we use the **mean hourly rate** within each zone. This gives a representative zone-level price, but it may hide more complex pricing rules such as different meter schedules, time-based rates, or special cases within the same zone.


#### 2.4. Code
The code here is designed to provide a clearer overview of the notebook. For more detailed, low-level information, please see ./detailed_analysis/price_busyness.ipynb.

**Variable Praparation**

In [ ]:
# ------------------------------------------------------------
# Keep only zones that can be joined across all required datasets
# ------------------------------------------------------------
rates = (
    parking_rates
    # Remove repeated pricing rules that appear across multiple meters
    .unique(subset=["zone_id", "rate_window", "rate_time", "rate"], keep="first")
    .join(zone_group_norm_to_zone_id, on="zone_id")
    .filter(pl.col("zone_group_norm").is_in(joinable_zones_lst))
)

tx = payment_transactions.filter(
    pl.col("zone_group_norm").is_in(joinable_zones_lst)
)

spaces = parking_spaces_zone_group_norm.filter(
    pl.col("zone_group_norm").is_in(joinable_zones_lst)
)


# ------------------------------------------------------------
# Clean pricing data
# From earlier analysis, we found that Oakland 3 has special
# time-specific rate rows in addition to "all time" rows.
# To keep one consistent zone-level rate, we drop the
# non-"all time" entries.
#
# In practice, this means that in Oakland 3 some meters have
# one constant rate for all times, while others have different
# prices for different time ranges.
# ------------------------------------------------------------
rates = rates.filter(
    ~(
        (pl.col("zone_group_norm") == "OAKLAND 3") &
        (pl.col("rate_time") != "all time")
    )
)


# ------------------------------------------------------------
# Compute average zone-level price
# Some zones still have multiple valid rate rows, so we aggregate
# them into a single representative zone price using the mean.
#
# In practice, this means that within a zone there may still be
# meters with different prices, so we use the average as a
# summary value for the zone.
# ------------------------------------------------------------
avg_rates = (
    rates
    .select(["zone_group_norm", "rate"])
    .group_by("zone_group_norm")
    .mean()
)


# ------------------------------------------------------------
# Assign each zone to a broad price category, since values like
# 1.25 and 2.25 are derived by averaging rates within a zone
# ------------------------------------------------------------
avg_rates = avg_rates.with_columns(
    pl.when(pl.col("rate").is_in([1.0, 1.25]))
      .then(pl.lit("Group 1: (1$, 1.25$)"))
    .when(pl.col("rate").is_in([1.5]))
      .then(pl.lit("Group 2: (1.5$)"))
    .when(pl.col("rate").is_in([2.0, 2.25]))
      .then(pl.lit("Group 3: (2$, 2.25$)"))
    .when(pl.col("rate").is_in([3.0, 4.0]))
      .then(pl.lit("Group 4: (3$, 4$)"))
    .alias("price_category")
)


# ------------------------------------------------------------
# Build zone-level busyness = transactions / spaces
# ------------------------------------------------------------
tx_by_zone = (
    tx
    .group_by("zone_group_norm")
    .agg(
        pl.col("n_transaction").sum().alias("transactions")
    )
)

busyness = (
    tx_by_zone
    .join(spaces, on="zone_group_norm", how="inner")
    .with_columns(
        (pl.col("transactions") / pl.col("spaces")).alias("transactions_per_space")
    )
)


# ------------------------------------------------------------
# Combine zone-level price and busyness into one table
# This is the core table for the analysis.
# ------------------------------------------------------------
price_busyness = (
    busyness
    .join(avg_rates, on="zone_group_norm", how="inner")
)

**Price & Busyness Relationship Plots**

In [ ]:
# ------------------------------------------------------------
# Summarize busyness by price category
# Useful for seeing whether more expensive categories are also
# busier on average.
# ------------------------------------------------------------
by_price_category = (
    price_busyness
    .group_by("price_category")
    .agg([
        pl.col("transactions_per_space").mean().alias("mean_busyness"),
        pl.col("transactions_per_space").std().alias("std_busyness"),
        pl.col("spaces").mean().alias("mean_spaces"),
        pl.col("spaces").std().alias("std_spaces"),
        pl.col("zone_group_norm").n_unique().alias("n_zones"),
    ])
    .sort("price_category")
)

# average busyness by price level
avg_tx_space_bars = alt.Chart(by_price_category.to_pandas()).mark_bar().encode(
    x=alt.X("price_category:N",
            title="Level of Price", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("mean_busyness:Q", title="Average transactions per space"),
    color=alt.Color("mean_busyness:Q"),
    tooltip=[
        "price_category:O",
        alt.Tooltip("mean_busyness:Q", format=".3f"),
        alt.Tooltip("std_busyness:Q", format=".3f"),
        "n_zones:Q",
    ],
).properties(
    title="Figure 2.1 — Mean busyness by price level",
    width=400,
    height=380,
)

# --------------------------------------------------------------------
# Scatter + regression: each point is a zone.
# It shows how busyness (transactions per space) changes with price,
# while the regression line shows the overall trend.
# --------------------------------------------------------------------
avg_tx_space_scatter = alt.Chart(price_busyness.to_pandas()).mark_circle(size=70, opacity=0.6).encode(
    x=alt.X("rate:Q",
            title="Hourly rate ($/hour)", scale=alt.Scale(domain=[0.5, 4.5]),),
    y=alt.Y("transactions_per_space:Q", title="Average transactions per space"),
    tooltip=[
        "zone_group_norm:N",
        alt.Tooltip("rate:Q", format=".2f"),
        alt.Tooltip("transactions_per_space:Q", format=".3f"),
        "price_category:N",
        alt.Tooltip("spaces:Q", format=",.0f")
    ]
).properties(width=400, height=380, title="Figure 2.2 — Price-busyness scatter with regression trend")

# regression trend line, it shows the overall trend between parking price and busyness
trend = alt.Chart(price_busyness.to_pandas()).transform_regression(
    "rate", "transactions_per_space"
).mark_line().encode(
    x="rate:Q",
    y="transactions_per_space:Q"
)

# boxplot to compare the distribution of busyness
avg_tx_space_box = (
    alt.Chart(price_busyness.to_pandas())
    .mark_boxplot(size=45)
    .encode(
        # x-axis: categorical price groups
        x=alt.X(
            "price_category:N",
            title="Level of Price",
            axis=alt.Axis(labelAngle=0)  # keep labels horizontal
        ),
        # y-axis: zone-level busyness proxy
        y=alt.Y(
            "transactions_per_space:Q",
            title="Transactions per space"
        ),
        # tooltip: show the price group when hovering
        tooltip=[
            alt.Tooltip("price_category:N", title="Price level")
        ]
    )
    .properties(
        title="Figure 2.3 - Distribution of busyness by price level",
        width=400,
        height=380
    )
)


# Display the chart
avg_tx_space_bars | avg_tx_space_scatter + trend

alt.HConcatChart(...)

In [ ]:
# Boxplot display
avg_tx_space_box

alt.Chart(...)

In [ ]:
# ------------------------------------------------------------
# Distribution of zones across price categories
# This shows how many zones fall into each price level.
# ------------------------------------------------------------
avg_rate_dist = (
    avg_rates
    .group_by("price_category")
    .len()
    .rename({"len": "count"})
    .with_columns(
        # convert counts to percentages for optional reporting
        (pl.col("count") / pl.col("count").sum() * 100).alias("percent")
    )
    .sort("price_category")
)

rate_bars_grouped = (
    alt.Chart(avg_rate_dist.to_pandas())
    .mark_bar()
    .encode(
        # x-axis: grouped hourly price categories
        x=alt.X(
            "price_category:O",
            title="Rate ($/hour)",
            axis=alt.Axis(labelAngle=0)
        ),
        # y-axis: number of zones in each category
        y=alt.Y(
            "count:Q",
            title="Number of zones"
        ),
        # color bars by count for easier visual comparison
        color=alt.Color(
            "count:Q",
            scale=alt.Scale(scheme="purples"),
            title="Number of zones"
        ),
        # tooltip for exact values on hover
        tooltip=[
            alt.Tooltip("price_category:O", title="Rate"),
            alt.Tooltip("count:Q", title="N_zone"),
            alt.Tooltip("percent:Q", format=".1f", title="Percent")
        ],
    )
    .properties(
        title="Figure 2.4 — Distribution of zones and average spaces by price level",
        width=400,
        height=320,
    )
)

# ------------------------------------------------------------
# Average spaces by price category
# This helps check whether more expensive zones also tend to
# have larger or smaller parking supply.
# ------------------------------------------------------------
avg_spaces_bars = (
    alt.Chart(by_price_category.to_pandas())
    .mark_bar()
    .encode(
        # x-axis: price category
        x=alt.X(
            "price_category:N",
            title="Level of Price",
            axis=alt.Axis(labelAngle=0)
        ),
        # y-axis: average number of spaces across zones in that category
        y=alt.Y(
            "mean_spaces:Q",
            title="Average spaces"
        ),
        # color by average spaces for visual emphasis
        color=alt.Color(
            "mean_spaces:Q",
            title="Average spaces"
        ),
        # tooltip with summary statistics
        tooltip=[
            "price_category:O",
            alt.Tooltip("mean_spaces:Q", format=".3f", title="Mean spaces"),
            alt.Tooltip("std_spaces:Q", format=".3f", title="Std spaces"),
            alt.Tooltip("n_zones:Q", title="N_zone"),
        ],
    )
    .properties(
        title="Figure 2.5 — Distribution of spaces within each price level",
        width=400,
        height=320,
    )
)

# Charts display
(
    (rate_bars_grouped | avg_spaces_bars)
    .resolve_scale(color="independent")
    .resolve_axis(y="independent")
)

alt.HConcatChart(...)

In [ ]:
# boxplot to compare the distribution of parking supply
box_spaces = (
    alt.Chart(price_busyness.to_pandas())
    .mark_boxplot(size=40)
    .encode(
        # x-axis: categorical price groups
        x=alt.X(
            "price_category:N",
            title="Price level",
            axis=alt.Axis(labelAngle=0),  # keep labels horizontal
        ),
        # y-axis: number of spaces available in each zone
        y=alt.Y(
            "spaces:Q",
            title="Spaces per zone"
        ),
        # tooltip: show the price category when hovering
        tooltip=[
            alt.Tooltip("price_category:N", title="Price level")
        ],
    )
    .properties(
        title="Figure 2.5 — Spaces per zone by price level",
        width=650,
        height=380,
    )
)

# display the chart
box_spaces

alt.Chart(...)

In [ ]:
# ------------------------------------------------------------
# Focus on the middle price groups (Groups 2 and 3)
# These are the most interesting for mismatch analysis because
# they have great standard devation.
# ------------------------------------------------------------
target_groups = ["Group 2: (1.5$)", "Group 3: (2$, 2.25$)"]

g23 = price_busyness.filter(
    pl.col("price_category").is_in(target_groups)
)


# ------------------------------------------------------------
# Compute within-group thresholds
# We use the 10th and 90th percentiles of transactions_per_space
# to identify unusually under-busy / over-busy zones relative to
# other zones in the same price band.
# ------------------------------------------------------------
thr = (
    g23
    .group_by("price_category")
    .agg([
        pl.col("transactions_per_space").quantile(0.10).alias("q10"),
        pl.col("transactions_per_space").quantile(0.90).alias("q90"),
        pl.col("transactions_per_space").median().alias("median"),
        pl.col("transactions_per_space").len().alias("n_zones"),
    ])
)


# ------------------------------------------------------------
# Label mismatch zones
# Over-busy  = top 10% busyness within the price group
# Under-busy = bottom 10% busyness within the price group
# ------------------------------------------------------------
mismatch = (
    g23
    .join(thr, on="price_category", how="left")
    .with_columns(
        pl.when(pl.col("transactions_per_space") >= pl.col("q90"))
          .then(pl.lit("Over-busy (top 10%)"))
        .when(pl.col("transactions_per_space") <= pl.col("q10"))
          .then(pl.lit("Under-busy (bottom 10%)"))
        .otherwise(pl.lit("Typical"))
        .alias("mismatch_type")
    )
    .filter(pl.col("mismatch_type") != "Typical")
    .select([
        "price_category",
        "zone_group_norm",
        "rate",
        "transactions_per_space",
        "spaces",
        "mismatch_type",
    ])
    .sort(["price_category", "transactions_per_space"], descending=[False, True])
)


# ------------------------------------------------------------
# Measure how far each mismatch zone is from its group median
# This gives a cleaner way to visualize the severity of mismatch.
# ------------------------------------------------------------
mm_dev = (
    g23
    .join(thr, on="price_category", how="left")
    .with_columns([
        (pl.col("transactions_per_space") - pl.col("median")).alias("dev_from_median"),
        pl.when(pl.col("transactions_per_space") >= pl.col("q90"))
          .then(pl.lit("Over-busy"))
          .when(pl.col("transactions_per_space") <= pl.col("q10"))
          .then(pl.lit("Under-busy"))
          .otherwise(pl.lit(None))
          .alias("mismatch_type"),
    ])
    .filter(pl.col("mismatch_type").is_not_null())
    .select([
        "price_category",
        "zone_group_norm",
        "rate",
        "spaces",
        "transactions_per_space",
        "median",
        "dev_from_median",
        "mismatch_type",
    ])
    .sort(["price_category", "dev_from_median"], descending=[False, True])
)


# ------------------------------------------------------------
# Convert to pandas for Altair plotting
# ------------------------------------------------------------
mm_dev_pd = mm_dev.to_pandas()

g2_label = "Group 2: (1.5$)"
g3_label = "Group 3: (2$, 2.25$)"

g2 = mm_dev_pd[mm_dev_pd["price_category"] == g2_label]
g3 = mm_dev_pd[mm_dev_pd["price_category"] == g3_label]


# ------------------------------------------------------------
# Plot mismatch severity within each price group
# Positive deviation = busier than the group median
# Negative deviation = less busy than the group median
# ------------------------------------------------------------
def mismatch_bar(df, title):
    """
    Create a bar chart showing how far each mismatch zone deviates
    from the median busyness of its price category.

    Parameters
    df : pandas.DataFrame
        Subset of mismatch zones for one price category.
    title : str
        Chart title.

    Returns
    alt.Chart
        Altair bar chart of deviation from group median.
    """
    return (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X(
                "dev_from_median:Q",
                title="Deviation from group median (tx/space)"
            ),
            y=alt.Y(
                "zone_group_norm:N",
                sort="-x",
                title="Zone"
            ),
            color=alt.Color(
                "dev_from_median:Q",
                title="Mismatch"
            ),
            tooltip=[
                "zone_group_norm:N",
                alt.Tooltip("rate:Q", format=".2f", title="Rate"),
                alt.Tooltip("transactions_per_space:Q", format=".3f", title="Tx/space"),
                alt.Tooltip("median:Q", format=".3f", title="Group median"),
                alt.Tooltip("dev_from_median:Q", format=".3f", title="Deviation"),
                alt.Tooltip("spaces:Q", format=",.0f", title="Spaces"),
                "mismatch_type:N",
            ],
        )
        .properties(width=330, height=380, title=title)
    )


c1 = mismatch_bar(g2, "Figure 2.6a — Group 2 mismatch zones relative to group median")
c2 = mismatch_bar(g3, "Figure 2.6b — Group 3 mismatch zones relative to group median")

(c1 | c2).resolve_scale(x="shared", color="shared")

alt.HConcatChart(...)

### 2.5. Interpretation

The main takeaway is:
> **prices are generally related to busyness, but the fit is imperfect, especially in the middle price tiers.**


Overall, the figures suggest a **positive relationship between parking price and busyness**. In **Figure 2.1**, average transactions per space rise clearly from **Group 1** to **Group 3** this indicates that **higher-priced zones are generally busier**, so pricing is at least broadly aligned with demand. That said, the pattern is **not perfectly monotonic**. In **Figure 2.1**, **Group 4 (\$3-\$4)** is slightly **less busy on average than Group 3 (\$2-\$2.25)**. However, this does not overturn the overall conclusion, because the scatter in **Figure 2.2** still shows an upward overall trend.

A more likely explanation is that **Group 4 is structurally different** from the other groups rather than simply "mispriced". This interpretation is supported by **Figure 2.4** and **Figure 2.5**. Group 4 contains **fewer zones**, and those zones tend to have **much larger parking supply**. That makes them less directly comparable to the other price tiers. These may be zones with different parking functions, such as larger lots, longer-duration stays, lease or permit-oriented use, or higher operating and maintenance costs. In that case, the city may not be able to set Group 4 prices purely according to short-term demand, and the observed price level may reflect those structural features.

**Key Insight**

The clearest sign that pricing is **not perfectly aligned** with demand comes from **Figure 2.3**. The boxes for **Group 2** and **Group 3** are very wide, which means that zones within the same price tier have **very different levels of busyness**. So even though price and busyness move together on average, many individual zones inside the same tier behave quite differently. This suggests that the current grouped pricing system captures the broad pattern, but still leaves substantial room for mismatch.

That mismatch becomes more concrete in **Figure 2.6a** and **Figure 2.6b**. Some zones are much **busier than the median** for their price group, while others are much **less busy than the median**. In practical terms, this means some zones are likely **underpriced relative to demand**, while others may be **overpriced**.

#### Action to recommend

A good next step would be to move toward a **more demand-responsive pricing review**, especially for **Groups 2 and 3**, since those tiers show the widest within-group variation. The city could begin by examining the mismatch zones in **Figure 2.6a** and **Figure 2.6b**:
- zones that are **too busy for their price level** could be candidates for a moderate price increase
- while zones that are **too quiet for their price level** could be candidates for a reduction

At the same time, **Group 4 should probably be treated separately**. Because those zones appear to be fewer, larger, and potentially serving different parking purposes, their pricing may reflect more than just observed transaction demand. Any pricing adjustment there should consider operational cost, long-stay behavior, lease use, and local context.

## Question 3: Which parking areas generate the least revenue per parking space?

### 3.1 Why this question is valuable?
Oftentimes, we may encounter situations where you arrive at a parking lot but do not have enough space. This question helps us resolve this problem by identifying parking areas that have limited space with high demand as well as abundant space with limited demand. This will be valuable for the department of urban planning to expand certain parking regions and provide more convenient parking services to our citizens.

Another crucial point is that there are a lot of people might forget or do not pay the parking fees intentionally. Despite the datasets do not include all citations that we have in history, we believe this analysis could be a great foundation to evaluate whether certain areas should be our target if we want to investigate in this problem in the future. This will be valuable to the government as it is related to maximizing government revenus.

### 3.2 Method
We did two baisc exploratory analysis, where we look at the top 10 parking zones in terms of total transactions in history. We also used another bar plot to check when do parking demand reaches its peak during a random day. Everythign seems to be resonated to our hypothesis and assumptions. Therefore, we eventually chose to look at parking areas that generate the least revenue. In the third visuallization, we combine parking transaction data with parking supply data to measure how much revenue each parking space generates on average. This allows us to compare parking areas of different sizes in a fair way.

Parking transaction records provide information on total payments and usage, while the parking spaces dataset provides the number of spaces available in each area. These datasets are joined using the parking zone identifier so that revenue and demand can be evaluated relative to parking supply.

To ensure the analysis focuses on areas with meaningful activity, we filter out locations with extremely low parking usage (i.e. minimum transactions per space per day must be larger or equal to \$0.25). This prevents rarely used parking areas from appearing artificially inefficient.

### 3.3 Assumptions and Limitations
We assume downtown areas to be considered having the most total transactions since it's where traffic occurss the most. Another limitation is that we can only infer the problem of people not paying parking fees from those areas that generate least revenues; however, we do not have sufficient data to prove it's true. This is why we propose that in the future, given enough time and resources, the government should record the number of citations so that we can analyze where most violations occur and use that as a guidance to decide whether we should have more people checking on payment and so on.

### 3.4 Code

In [ ]:
import polars as pl
import math
import altair as alt

# path = '/content/drive/MyDrive/95885 - Project (Andrew Huang, Bohua Liu)/final_datasets/'
path = 'final_datasets/'

# Load
parking_zones = pl.read_csv(f"{path}parking_zones.csv")
zone_group_norm_to_zone_id = pl.read_csv(f"{path}zone_group_norm_to_zone_id.csv")
parking_meters = pl.read_csv(f"{path}parking_meters.csv")
parking_rates = pl.read_csv(f"{path}parking_rates.csv")
parking_rates = (
    parking_rates
    # Remove repeated pricing rules that appear across multiple meters
    .unique(subset=["zone_id", "rate_window", "rate_time", "rate"], keep="first")
)
parking_spaces = pl.read_csv(f"{path}parking_spaces_zone_group_norm.csv")
payment_transactions = pl.read_csv(f"{path}payment_transactions.csv")

# Parse timestamps
payment_transactions = payment_transactions.with_columns([
    pl.col("start").str.strptime(pl.Datetime, strict=False),
    pl.col("end").str.strptime(pl.Datetime, strict=False),
    pl.col("utc_start").str.strptime(pl.Datetime, strict=False)
])

### Exploratory Data Analysis

In [ ]:
tx_by_zone = (
    payment_transactions
    .group_by("zone_group_norm")
    .agg(pl.col("n_transaction").sum().alias("transactions"))
)

top10_zone_with_most_tx = (
    tx_by_zone
    .sort("transactions", descending=True)
    .head(10)
)

chart = alt.Chart(top10_zone_with_most_tx).mark_bar().encode(
    x=alt.X("transactions:Q", title="Number of transactions"),
    y=alt.Y("zone_group_norm:N", sort="-x", title="Zone"),
    tooltip=["zone_group_norm:N", "transactions:Q"]
).properties(
    title="Top 10 Zones by Number of Transactions",
    width=700,
    height=350
)

chart

alt.Chart(...)

### Interpretation of visualization 1.
From the bar plot above (i.e. Top 10 Zones by Number of Transactions), we first identified those that have the highest demand in history. The result resonates with our assumption because the first three appears to be southside work, squirrel hills, and strip district. All these three areas are where restaurants, small shopping stores, and coffee shops are located at. They have their own business offices as well as residential apartments, where a lot of local residents would spend their whole afternoon. This might be the reason they would pay for the parking and have such as high demand. In particular, if you go to southside, you will see a variety of shops including bars, various types of cuisines, and even grocery stores. So we also hypothesize that southside would continue to be the top 1 area to have the highest number of transactions. Something suprising is that downtown shows up as number 8 and 9 in the plot. We assumed downtown to be around one of the top 5 areas as it has the most number of vehicles going through and so many people work around the region. We then brainstormed and did some research and made an important assumption that people who visit downtown tend to stay for a short amount time, which makes them less likely to pay for the parking. Additionally, due to the limited space and the amount of cars, people tend to have a psychological mindset thinking they will not be unlucky enough to get caught.

### Interpretation of visualization 2.
For the bar plot below, we created a bar plot that shows the hourly average transactions during a day. It shows a typical normal distribution, where the number of transactions reaches its peak at noon and the number reaches its lowest point at the tails (either in the very early morning or in the late night). It's uncertain why the trend would look like this because we all believe that people that go to work pay in the morning, so we assumed morning transactions should be higher. We assume it's because these transactions are usually paid by people that only stay or park for a short amount of time, which is why during lunch breaks, we have a much higher demand than other times during a day.

In [ ]:
tx_by_hour = (
    payment_transactions
    .with_columns(
        pl.col("start").dt.hour().alias("hour")
    )
    .group_by("hour")
    .agg([
        pl.col("n_transaction").mean().alias("avg_total_txn")
    ])
    .sort("hour")
)

# Avg total transactions amount per 10-min bin
alt.Chart(tx_by_hour.to_arrow()).mark_bar().encode(
    x=alt.X("hour:O", title="Hour of day"),
    y=alt.Y("avg_total_txn:Q", title="Avg total transactions per 10-min bin"),
    tooltip=["hour:O", alt.Tooltip("avg_total_txn:Q", title="average total transactions",format=".2f")],
)

alt.Chart(...)

The code below focuses on identifying parking lots in Pittsburgh that generate the least revenues. This question is essential not only in identifying parking areas that are not generating revenues, but also is a potential indicator of whether people do not pay when they park in certain areas. The metrics computed are per-space metrics, meaning we look at the revenue generated from each parking space within each parking area within a day. This helps us avoids confounding variables such as the size of the parking lot.



In [ ]:
# ---------------------------------------------------------
# 1. Parameters
# ---------------------------------------------------------
min_transactions_per_space_per_day = 0.25
top_n_areas = 15

# ---------------------------------------------------------
# 2. Build lookup table for area names and full locations
# ---------------------------------------------------------
zone_lookup = (
    parking_zones
    .select(["zone_group_norm", "location_list"])
    .unique(subset=["zone_group_norm"], keep="first")
    .with_columns([
        pl.col("zone_group_norm").alias("area_name"),
        pl.col("location_list").fill_null(pl.col("zone_group_norm")).alias("location_full")
    ])
)

# ---------------------------------------------------------
# 3. Build parking supply table
#    (Sum spaces in case the same area appears more than once)
# ---------------------------------------------------------
spaces_by_area = (
    parking_spaces
    .group_by("zone_group_norm")
    .agg([
        pl.col("spaces").sum().alias("spaces"),
        pl.col("type").alias("parking_type")
    ])
)

# ---------------------------------------------------------
# 4. Aggregate parking payment activity by area
# ---------------------------------------------------------
revenue_by_area = (
    payment_transactions
    .with_columns(
        pl.col("start").dt.date().alias("date")
    )
    .group_by("zone_group_norm")
    .agg([
        pl.col("amount").sum().alias("total_revenue"),
        pl.len().alias("total_transactions"),
        pl.col("date").n_unique().alias("n_days_observed")
    ])
)

# ---------------------------------------------------------
# 5. Join revenue, supply, and area metadata
#    Then compute per-space metrics
# ---------------------------------------------------------
revenue_space_analysis = (
    revenue_by_area
    .join(spaces_by_area, on="zone_group_norm", how="left")
    .join(zone_lookup, on="zone_group_norm", how="left")
    .filter(
        pl.col("spaces").is_not_null() & (pl.col("spaces") > 0)
    )
    .with_columns([
        (pl.col("total_revenue") / pl.col("spaces")).alias("revenue_per_space"),
        (pl.col("total_transactions") / pl.col("spaces")).alias("transactions_per_space"),
        (pl.col("total_revenue") / pl.col("n_days_observed")).alias("avg_daily_revenue"),
        (
            pl.col("total_revenue") /
            (pl.col("spaces") * pl.col("n_days_observed"))
        ).alias("avg_daily_revenue_per_space"),
        (
            pl.col("total_transactions") /
            (pl.col("spaces") * pl.col("n_days_observed"))
        ).alias("transactions_per_space_per_day")
    ])
    .with_columns([
        pl.col("total_revenue").round(2),
        pl.col("revenue_per_space").round(2),
        pl.col("transactions_per_space").round(2),
        pl.col("avg_daily_revenue").round(2),
        pl.col("avg_daily_revenue_per_space").round(3),
        pl.col("transactions_per_space_per_day").round(3)
    ])
)

# ---------------------------------------------------------
# 6. Filter out very low-demand areas
#    This keeps only areas with meaningful parking activity
# ---------------------------------------------------------
analysis_filtered = (
    revenue_space_analysis
    .filter(
        pl.col("transactions_per_space_per_day") >= min_transactions_per_space_per_day
    )
)

# ---------------------------------------------------------
# 7. Keep the lowest-revenue areas for the chart
# ---------------------------------------------------------
bar_df = (
    analysis_filtered
    .select([
        "area_name",
        "location_full",
        "parking_type",
        "spaces",
        "total_revenue",
        "total_transactions",
        "n_days_observed",
        "revenue_per_space",
        "transactions_per_space",
        "transactions_per_space_per_day",
        "avg_daily_revenue_per_space"
    ])
    .sort("avg_daily_revenue_per_space")
    .head(top_n_areas)
)

bar_data = bar_df.to_dicts()

# Rows used in the chart
bar_df

area_name,location_full,parking_type,spaces,total_revenue,total_transactions,n_days_observed,revenue_per_space,transactions_per_space,transactions_per_space_per_day,avg_daily_revenue_per_space
str,str,list[str],i64,f64,u32,u32,f64,f64,f64,f64
"""OBERSERVATORY HILL LOT""","""OBSERVATORY HILL LOT""","[""off-street""]",23,4572.03,6455,778,198.78,280.65,0.361,0.256
"""BROWNSVILLE SANDKEY LOT""","""BROWNSVILLE SANKEY LOT""","[""off-street""]",80,34131.01,22286,971,426.64,278.58,0.287,0.439
"""WALTER WARRINGTON LOT""","""WALTER WARRINGTON LOT""","[""off-street""]",15,3498.94,2449,491,233.26,163.27,0.333,0.475
"""WEST END""","""MAIN ST""","[""on-street""]",52,16973.21,11446,507,326.41,220.12,0.434,0.644
"""BEECHVIEW LOT""","""BEECHVIEW LOT""","[""off-street""]",17,6412.55,3439,584,377.21,202.29,0.346,0.646
…,…,…,…,…,…,…,…,…,…,…
"""HILL DISTRICT""",""" ROBINSON ST, ROBINSON ST EXT…","[""on-street""]",26,13948.48,8616,492,536.48,331.38,0.674,1.09
"""SHERIDAN HARVARD LOT""","""SHERIDAN HARVARD LOT""","[""off-street""]",41,51107.34,15175,967,1246.52,370.12,0.383,1.289
"""EVA BEATTY LOT""","""EVA BEATTY LOT""","[""off-street""]",130,182203.67,33385,971,1401.57,256.81,0.264,1.443


In [ ]:
# ---------------------------------------------------------
# 8. Visualization: lowest average daily revenue per space
# ---------------------------------------------------------
bar_chart = (
    alt.Chart(alt.Data(values=bar_data))
    .mark_bar()
    .encode(
        x=alt.X(
            "avg_daily_revenue_per_space:Q",
            title="Average Daily Revenue per Space ($)",
            axis=alt.Axis(labelFontSize=12, titleFontSize=14)
        ),
        y=alt.Y(
            "area_name:N",
            sort=alt.SortField(field="avg_daily_revenue_per_space", order="ascending"),
            title="Parking Area",
            axis=alt.Axis(labelFontSize=12, titleFontSize=14)
        ),
        color=alt.Color(
            "avg_daily_revenue_per_space:Q",
            title="Avg Daily Revenue per Space",
            scale=alt.Scale(scheme="blues"),
            legend=alt.Legend(orient="bottom")
        ),
        tooltip=[
            alt.Tooltip("area_name:N", title="Area"),
            alt.Tooltip("location_full:N", title="Full Location"),
            alt.Tooltip("parking_type:N", title="Parking Type"),
            alt.Tooltip("spaces:Q", title="Spaces"),
            alt.Tooltip("total_revenue:Q", title="Total Revenue ($)", format=".2f"),
            alt.Tooltip("total_transactions:Q", title="Transactions"),
            alt.Tooltip("n_days_observed:Q", title="Observed Days"),
            alt.Tooltip("revenue_per_space:Q", title="Revenue per Space ($)", format=".2f"),
            alt.Tooltip("transactions_per_space:Q", title="Transactions per Space", format=".2f"),
            alt.Tooltip(
                "transactions_per_space_per_day:Q",
                title="Transactions per Space per Day",
                format=".3f"
            ),
            alt.Tooltip(
                "avg_daily_revenue_per_space:Q",
                title="Avg Daily Revenue per Space ($)",
                format=".3f"
            )
        ]
    )
    .properties(
        width=850,
        height=450,
        title=(
            "Parking Areas Generating the Least Average Daily Revenue per Space\n"
            f"(Transactions per Space per Day ≥ {min_transactions_per_space_per_day})"
        )
    )
)

bar_chart

alt.Chart(...)

### Interpretation of visualization 3 (above)
The visualization compares parking areas based on their average daily revenue generated per parking space. The chart shows that several locations generate substantially less revenue per space than others, even after filtering for areas with measurable parking demand.

For example, Observatory Hill Lot produces only about $0.26 per space per day, the lowest among the analyzed locations. Brownsville Sandkey Lot and Walter Warrington Lot also generate relatively low revenue per space.

In contrast, areas such as Ansley Beatty Lot, Mt. Washington, and Eva Beatty Lot generate significantly higher revenue per space. Interestingly, several of these higher-performing areas have relatively small numbers of parking spaces, such as Ansley Beatty Lot with 23 spaces and Hill District with 26 spaces. This suggests that limited supply may concentrate demand and lead to higher utilization per space.

Another pattern is that many higher-performing locations are off-street parking lots, while some lower-performing areas include both off-street and on-street parking zones. This indicates that parking type and supply size may both influence revenue efficiency.


### Recommendation
First, areas with high revenue per space and limited parking supply, such as Ansley Beatty Lot and Hill District, may indicate strong demand relative to available parking. The city could consider expanding parking capacity in these locations or improving parking accessibility nearby to better accommodate demand.

Second, areas with very low revenue per space, such as Observatory Hill Lot and Brownsville Sandkey Lot, may have excess parking supply relative to demand. In these areas, the city may consider reducing parking capacity, repurposing some parking spaces for alternative uses, or reevaluating pricing policies to better reflect local demand conditions.

Third, the city may consider reviewing pricing policies across parking types, as off-street parking appears to generate higher revenue per space in several locations. Adjusting pricing structures or time limits could help improve utilization efficiency.

### Future Work
Based on this analysis, we have found several interesting patterns and trends. Given sufficient datasets and time, here are some additional work that should be considered in the future.

The first step is to analyze time-based parking demand patterns, such as variations in parking usage by hour of the day or day of the week. This could reveal whether some locations experience peak demand periods that are not visible in aggregated data.

Another important area of investigation is parking compliance behavior. Some drivers may park without paying for parking meters, which would reduce recorded revenue even if spaces are being used. Incorporating data on parking violations or enforcement activity could help determine whether low-revenue areas are due to low demand or unpaid parking usage.

Finally, future analysis could incorporate additional contextual factors such as nearby commercial activity, population density, and public transportation availability. These factors may help explain differences in parking demand across neighborhoods and support more data-driven parking policy decisions.